# ATML PA2 - Task 4: Safety calibration

**Before running:** Settings → Accelerator **GPU T4** (x1 or x2), Internet **On**.

Run **after** Tasks 1-3. Needs the three standard adapters. Upload `task1_outputs.zip`, `task2_outputs.zip`, `task3_outputs.zip` as one Kaggle Dataset and add it to this notebook (Add Input). Rough T4 time: ~1.5-2 h.

## 1. Setup

In [ ]:
# Fresh session: clone; re-run: pull the latest code.
%cd /kaggle/working
!if [ -d pa2 ]; then git -C pa2 pull; else git clone -b pa2 https://github.com/rana-taqveem/atml_internal.git pa2; fi
%cd /kaggle/working/pa2
!pip install -q -r requirements.txt
!python -m scripts.download_assets
!python -m scripts.validate_assets

In [ ]:
!python -m scripts.check_environment   # expect CUDA available: True, GPU: Tesla T4

## 2. Bring in the standard DPO / PPO / GRPO adapters

In [ ]:
# Unzips every task*_outputs.zip found under /kaggle/input (any dataset name).
!find /kaggle/input -name 'task*_outputs.zip' -print -exec unzip -oq {} -d /kaggle/working/pa2 \;
!ls outputs/task1_dpo/standard outputs/task2_ppo/standard outputs/task3_grpo/standard

## 3. Step 1 - deterministic responses from SFT, DPO, PPO, GRPO (450 XSTest prompts each)

In [ ]:
!python -m task4_safety.generate_responses --config configs/feedback.yaml --skip-existing 2>&1 | tee -a task4.log

## 4. Step 2 - categorical AI judge (cached; re-running resumes)

In [ ]:
!python -m task4_safety.judge_responses --config configs/feedback.yaml 2>&1 | tee -a task4.log

## 5. Step 3 - blinded manual-audit sheet + AI-label aggregation
Default `--mode all` = 60 IDs x 4 policies = 240 rows; use `--mode rotate` for 60 rows.
Label the sheet **locally** after downloading; do not open `judged_*.jsonl` first.

In [ ]:
!python -m task4_safety.make_audit_sheet --config configs/feedback.yaml 2>&1 | tee -a task4.log
!python -m task4_safety.evaluate_safety --config configs/feedback.yaml 2>&1 | tee -a task4.log

## 6. Save (download `task4_outputs.zip` from the Output panel)

In [ ]:
!zip -qr /kaggle/working/task4_outputs.zip results/task4_safety task4.log
!ls -lh /kaggle/working/task4_outputs.zip